# NB_patient_360

Builds `adb_caresyncs.gold.patient_360` (one row per patient) from silver per the STTM *Silver to Gold* v1.0: patients LEFT JOIN hospitals, insurance_providers and aggregated lab_results. Full rebuild via CREATE OR REPLACE TABLE AS SELECT.

In [0]:
%sql
-- Target : adb_caresyncs.gold.patient_360
-- Grain  : one row per patient (driving table silver.patients)
-- Load   : FULL rebuild; LEFT JOINs only so no patient is dropped
CREATE OR REPLACE TABLE adb_caresyncs.gold.patient_360
COMMENT 'Gold: one row per patient with hospital, insurer and lab test summary (STTM silver to gold v1.0)'
AS
WITH lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                                   AS total_tests,
    COUNT_IF(result_status = 'Abnormal')       AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')       AS critical_test_count,
    MAX(test_date)                             AS last_test_date
  FROM adb_caresyncs.silver.lab_results
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                              AS patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE)                                          AS date_of_birth,
  CAST(FLOOR(DATEDIFF(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender,
  p.city,
  p.state,
  CAST(p.registered_hospital_id AS INT)                                  AS hospital_id,
  h.hospital_name,
  CAST(p.insurance_provider_id AS INT)                                   AS insurance_provider_id,
  i.provider_name                                                        AS insurance_provider_name,
  CAST(COALESCE(l.total_tests, 0) AS INT)                                AS total_tests,
  CAST(COALESCE(l.abnormal_test_count, 0) AS INT)                        AS abnormal_test_count,
  CAST(COALESCE(l.critical_test_count, 0) AS INT)                        AS critical_test_count,
  CAST(l.last_test_date AS DATE)                                         AS last_test_date
FROM adb_caresyncs.silver.patients p
LEFT JOIN adb_caresyncs.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN adb_caresyncs.silver.insurance_providers i
  ON p.insurance_provider_id = i.insurance_provider_id
LEFT JOIN lab_agg l
  ON p.patient_id = l.patient_id;

In [0]:
# Return the gold table record count to the caller (e.g. ADF / job task value)
updated_df = spark.read.table("adb_caresyncs.gold.patient_360")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))